# Lección 3 — Métricas de evaluación

**Tema:** cómo se mide si un modelo de Machine Learning es bueno o malo. RMSE para regresión. Confusion matrix + accuracy / precision / recall / F1 para clasificación.

**Objetivos de esta lección:**
- Interpretar el RMSE de un modelo de regresión en el contexto del problema.
- Leer una **confusion matrix** y entender los 4 cuadrantes (TP, FP, FN, TN).
- Calcular **accuracy, precision, recall y F1** a partir de TP/FP/FN/TN.
- Elegir cuál de las 4 métricas priorizar según el costo de falsos positivos vs falsos negativos.

**Side-note importante:** en este curso usamos los nombres de métricas en inglés — `accuracy` y `precision` son cosas DISTINTAS en ML, pero ambas se traducen como "precisión" al español. Para no confundirnos, usamos los nombres originales.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EduPav/ai-learning-notebooks/blob/main/lessons/03-metricas-evaluacion/notebook.ipynb)

## Índice

1. Regresión + RMSE — predecir precio de viviendas en California
2. Clasificación + Confusion Matrix — detección de cáncer de mama
3. Accuracy, Precision, Recall y F1
4. La trampa del accuracy en datos desbalanceados

## Cómo usar este notebook

- **Runtime sugerido:** CPU estándar de Colab. Todo corre en segundos.
- **Tiempo estimado:** ~20 minutos si lo recorrés con calma mirando los outputs y haciendo los ejercicios.
- **Convención de celdas:** todas las celdas son ejecutables. Corrélas en orden con Shift+Enter y leé los outputs mientras avanzás.
- **Librerías:** este notebook usa `pandas`, `numpy`, `sklearn`, `matplotlib` y `seaborn`. Las vamos a aprender a usar más adelante. Por ahora el objetivo es leer los outputs e interpretar los resultados.

In [ ]:
# Celda de setup: ejecutá esta celda una sola vez antes de avanzar.
# Todas las librerías vienen pre-instaladas en Google Colab,
# así que NO hace falta hacer pip install.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
)

# Fijamos seed para reproducibilidad.
np.random.seed(42)

# Estilo de las visualizaciones.
sns.set_theme(style="white")

print("Setup completo. Todo listo para correr las demos.")

## 1. Regresión + RMSE: predecir precio de viviendas en California

**Idea central:** vamos a entrenar un modelo que predice el precio promedio de viviendas en distintos distritos de California, y vamos a aprender a MEDIR qué tan bueno es ese modelo usando una métrica llamada **RMSE**.

**¿Qué es RMSE?** (raíz del error cuadrático medio)

- Es un número que resume "qué tan lejos estuvieron las predicciones de la realidad".
- Fórmula: `RMSE = raíz_cuadrada( promedio( (real - predicho)^2 ) )`
- ¿Por qué se eleva al cuadrado? Para penalizar los errores grandes más que los chicos. Un error de 10 cuenta 100; un error de 100 cuenta 10.000.
- ¿Por qué se toma raíz cuadrada al final? Para que el número esté en las mismas unidades que el output (dólares, no dólares al cuadrado).

**Lo importante:** RMSE bajo = modelo preciso. RMSE alto = modelo que se equivoca mucho. Pero "alto" o "bajo" depende del rango de los valores que predecís.

In [ ]:
# Cargamos el dataset California Housing.
# Cada fila es un distrito de California; el target es el precio mediano de la casa
# en ese distrito (medido en cientos de miles de dólares).

data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["precio_100k_USD"] = data.target

print("Dimensiones del dataset:", df.shape)
df.head(8)

Cada fila es un distrito. Las columnas son features sobre ese distrito (ingreso medio, edad de las viviendas, habitaciones, etc.). La última columna `precio_100k_USD` es el precio mediano en CIENTOS de miles de dólares — entonces 2.5 quiere decir USD 250.000.

In [ ]:
# Entrenamos un modelo de regresión lineal y comparamos predicciones vs realidad.

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo = LinearRegression()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

# Mostramos 12 distritos: lo que dijo el modelo vs el precio real.
tabla = pd.DataFrame({
    "precio_real_USD": np.round(y_test[:12] * 100_000, 0),
    "precio_predicho_USD": np.round(y_pred[:12] * 100_000, 0),
    "error_USD": np.round((y_pred[:12] - y_test[:12]) * 100_000, 0),
})
tabla

**¿Qué mirar?**

Mirá la columna `error_USD`. En algunas filas el modelo se equivoca por menos de USD 10.000 (bien); en otras se equivoca por más de USD 100.000 (falló feo). El RMSE va a resumir TODOS esos errores en un solo número.

In [ ]:
# Calculamos el RMSE sobre todas las predicciones.

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
rmse_USD = rmse * 100_000

print(f"RMSE = {rmse:.4f} (en unidades del target)")
print(f"RMSE = USD {rmse_USD:,.0f} (en dólares)")
print()
print(f"Rango de precios reales: USD {y_test.min()*100_000:,.0f} a USD {y_test.max()*100_000:,.0f}")

**Interpretación:** el modelo se equivoca en promedio ~USD 70.000. Si las viviendas valen entre USD 15.000 y USD 500.000 (rango muy amplio), errar 70.000 es mucho — está entre el 14% y el 470% del precio según el distrito. El modelo es **mediocre**.

**Regla a recordar:** RMSE NUNCA se interpreta solo. Siempre se compara contra el rango del output. Un RMSE de 50 puede ser excelente o pésimo según de qué se trate.

### Ejercicio — interpretá el RMSE

Un modelo predice el precio de tablets. Las tablets cuestan entre USD 200 y USD 800. RMSE = 50 USD. ¿Qué tan bueno es el modelo?

_Pista: ¿qué porcentaje del rango representa el error? ¿Es aceptable para una tienda de tecnología?_

<details>
<summary>Resultado esperado</summary>

El rango del output es USD 600 (de 200 a 800). Un RMSE de USD 50 representa el **8,3% del rango** — un error relativamente pequeño. Para una tienda de tecnología que usa el modelo para gestión de inventario o recomendaciones de precio, ese nivel de error es **aceptable**.

</details>

## 2. Clasificación + Confusion Matrix: detección de cáncer de mama

**Idea central:** ahora vamos a entrenar un modelo que **clasifica** tumores en malignos o benignos a partir de medidas de imágenes. Para evaluarlo NO basta con "¿acertó o no?" — necesitamos saber QUÉ tipo de error comete. Para eso usamos la **confusion matrix**.

**¿Qué es una confusion matrix?**

Es una tabla de 2×2 (en clasificación binaria) que cruza lo que el modelo PREDIJO con lo que en realidad ERA. Tiene 4 cuadrantes:

- **True Negatives (TN):** predijo benigno, ERA benigno. Acierto.
- **False Positives (FP):** predijo maligno, ERA benigno. Falsa alarma.
- **False Negatives (FN):** predijo benigno, ERA maligno. **Error grave en cáncer.**
- **True Positives (TP):** predijo maligno, ERA maligno. Acierto.

**Importante:** los dos errores (FP y FN) NO son iguales. En cáncer, perder un caso real (FN) es mucho peor que asustar a un sano (FP).

In [ ]:
# Cargamos el dataset Breast Cancer Wisconsin (incluido en sklearn).
# Cada fila es un tumor; las features son medidas de la imagen del tumor.
# El target es 0 (maligno) o 1 (benigno).

cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df["diagnostico"] = [cancer.target_names[i] for i in cancer.target]

print("Dimensiones del dataset:", df.shape)
print()
print("Distribución de clases:")
print(df["diagnostico"].value_counts())
print()
df[["mean radius", "mean texture", "mean perimeter", "mean area", "diagnostico"]].head(6)

569 tumores en total, ~37% malignos y ~63% benignos. Hay 30 features por tumor (medidas de la imagen). Mostramos 4 columnas representativas para que veas la pinta del dataset.

In [ ]:
# Entrenamos un clasificador y obtenemos la confusion matrix.
# Convención: 1 = maligno (es el caso "positivo" porque es lo que queremos detectar).

X = cancer.data
y = (cancer.target == 0).astype(int)  # 1 si maligno, 0 si benigno

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Estandarizamos features para que LogisticRegression converja mejor.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train_s, y_train)
y_pred = modelo.predict(X_test_s)

# Construimos la confusion matrix.
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"True Negatives  (TN): {tn} → predijo benigno, ERA benigno")
print(f"False Positives (FP): {fp} → predijo maligno, ERA benigno (falsa alarma)")
print(f"False Negatives (FN): {fn} → predijo benigno, ERA maligno (ERROR GRAVE)")
print(f"True Positives  (TP): {tp} → predijo maligno, ERA maligno")

In [ ]:
# Visualizamos la confusion matrix como heatmap (mucho más legible).

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicho: BENIGNO", "Predicho: MALIGNO"],
    yticklabels=["Real: BENIGNO", "Real: MALIGNO"],
    cbar=False, annot_kws={"size": 22}, ax=ax,
)
ax.set_title("Confusion Matrix — detección de cáncer", fontsize=14)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

**¿Qué mirar en este heatmap?**

- **Diagonal principal** (arriba-izq y abajo-der): aciertos del modelo. Números grandes = bueno.
- **Anti-diagonal** (arriba-der y abajo-izq): errores. Números chicos = bueno.
- **Cuadrante crítico:** abajo-izquierda → era maligno y el modelo dijo benigno. Estos son los **False Negatives**, los pacientes que se irían a casa pensando que están sanos cuando NO lo están.

### Ejercicio — interpretá la confusion matrix

En este caso del cáncer, ¿qué es peor: un FP o un FN? ¿Por qué?

<details>
<summary>Resultado esperado</summary>

Un **FN** (False Negative) es peor: el modelo predijo "benigno" pero el tumor era maligno. Ese paciente se iría a casa sin tratamiento cuando necesita atención urgente. Un FP (falsa alarma) también tiene costo — estrés y más estudios — pero se detecta con un chequeo adicional. En medicina, el principio es: mejor errar del lado de la precaución.

</details>

**Importante:** la confusion matrix es lo CRUDO. No te dice "el modelo es 90% bueno". Te dice exactamente cuántos errores de cada tipo cometió. Para resumirla en un número usamos las 4 métricas que vienen ahora.

## 3. Derivar accuracy, precision, recall y F1 desde la confusion matrix

**Idea central:** las 4 métricas que más vas a ver en clasificación (accuracy, precision, recall, F1) son distintas formas de RESUMIR la confusion matrix en UN solo número. Cada una pone el foco en una pregunta diferente.

Vamos una por una.

### 3.1. Accuracy — "de todas las predicciones, ¿cuántas estuvieron bien?"

Fórmula: `(TP + TN) / (TP + TN + FP + FN)`

Es la métrica más intuitiva: aciertos sobre total. **Pero tiene una trampa famosa que vamos a ver al final.**

In [ ]:
# Calculamos accuracy a mano y verificamos con sklearn.

accuracy_manual = (tp + tn) / (tp + tn + fp + fn)
accuracy_sklearn = accuracy_score(y_test, y_pred)

print(f"Accuracy (calculada a mano):        {accuracy_manual:.4f}")
print(f"Accuracy (sklearn, para verificar): {accuracy_sklearn:.4f}")
print()
print(f"Interpretación: de cada 100 tumores, el modelo acierta {accuracy_manual*100:.1f}.")

### 3.2. Precision — "de los que el modelo predijo positivos, ¿cuántos eran realmente positivos?"

Fórmula: `TP / (TP + FP)`

Pregunta sobre **las predicciones positivas del modelo**. Si tu modelo dice "este tumor es maligno", ¿qué tan confiable es esa predicción?

Útil cuando el costo de un FALSO POSITIVO es alto (ej: filtros de spam — no querrías marcar un mail importante como spam).

In [ ]:
# Calculamos precision a mano y verificamos con sklearn.

precision_manual = tp / (tp + fp)
precision_sklearn = precision_score(y_test, y_pred)

print(f"Precision (calculada a mano):        {precision_manual:.4f}")
print(f"Precision (sklearn, para verificar): {precision_sklearn:.4f}")
print()
print(f"Interpretación: cuando el modelo dijo 'maligno', acertó el {precision_manual*100:.1f}% de las veces.")

### 3.3. Recall — "de los que ERAN positivos, ¿cuántos atrapamos?"

Fórmula: `TP / (TP + FN)`

Pregunta sobre **los positivos reales**. ¿Cuántos casos reales de cáncer atrapamos?

Útil cuando el costo de un FALSO NEGATIVO es alto (ej: detección de cáncer — no podemos perder un caso real).

In [ ]:
# Calculamos recall a mano y verificamos con sklearn.

recall_manual = tp / (tp + fn)
recall_sklearn = recall_score(y_test, y_pred)

print(f"Recall (calculado a mano):        {recall_manual:.4f}")
print(f"Recall (sklearn, para verificar): {recall_sklearn:.4f}")
print()
print(f"Interpretación: de cada 100 tumores REALMENTE malignos, el modelo atrapó {recall_manual*100:.1f}.")
print(f"Perdimos {(1-recall_manual)*100:.1f}% de los cánceres reales.")

### 3.4. F1 — "un balance entre precision y recall"

Fórmula: `2 * (precision * recall) / (precision + recall)` (esto se llama **media armónica**, NO es promedio simple).

F1 es alto solo si AMBAS (precision y recall) son altas. Si una de las dos es muy baja, F1 cae mucho. Útil cuando querés balance entre los dos tipos de error.

In [ ]:
# Calculamos F1 a mano y verificamos con sklearn.

f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual)
f1_sklearn = f1_score(y_test, y_pred)

print(f"F1 (calculado a mano):        {f1_manual:.4f}")
print(f"F1 (sklearn, para verificar): {f1_sklearn:.4f}")
print()
print("Si precision y recall son altos, F1 es alto.")
print("Si una de las dos es muy baja, F1 lo refleja inmediatamente (a diferencia del promedio simple).")

### Resumen de las 4 métricas en este modelo

In [ ]:
# Tabla resumen de las 4 métricas calculadas sobre el modelo de cáncer.

resumen = pd.DataFrame({
    "métrica": ["Accuracy", "Precision", "Recall", "F1"],
    "fórmula": [
        "(TP + TN) / total",
        "TP / (TP + FP)",
        "TP / (TP + FN)",
        "2*(P*R) / (P+R)",
    ],
    "valor": [accuracy_manual, precision_manual, recall_manual, f1_manual],
    "pregunta que responde": [
        "de todas las predicciones, ¿cuántas estuvieron bien?",
        "cuando dije positivo, ¿cuántas veces acerté?",
        "de los positivos reales, ¿cuántos atrapé?",
        "¿qué tan balanceado está el modelo?",
    ],
})
resumen.style.format({"valor": "{:.4f}"})

### Ejercicio integrador 1 — calculá las 4 métricas desde cero

Un modelo de spam clasifica 100 mails. La confusion matrix es:

| | Predicho: NO spam | Predicho: spam |
|---|---|---|
| **Real: NO spam** | 60 (TN) | 5 (FP) |
| **Real: spam** | 10 (FN) | 25 (TP) |

Calculá accuracy, precision y recall usando las fórmulas de esta sección.

<details>
<summary>Resultado esperado</summary>

- **Accuracy** = (25 + 60) / (25 + 60 + 5 + 10) = 85 / 100 = **0.85 (85%)**
- **Precision** = 25 / (25 + 5) = 25 / 30 ≈ **0.833 (83.3%)**
- **Recall** = 25 / (25 + 10) = 25 / 35 ≈ **0.714 (71.4%)**

El modelo tiene buena accuracy y precision, pero su recall indica que perdió el 28.6% de los spams reales — esos terminaron en la bandeja de entrada.

</details>

### Ejercicio integrador 2 — elegí la métrica correcta

Para cada uno de estos escenarios, decidí cuál métrica (accuracy, precision, recall o F1) priorizarías y por qué:

a) Un modelo que detecta si hay incendio en imágenes de cámaras de seguridad de un edificio.  
b) Un modelo de recomendación de contenido que tiene que evitar tanto recomendar títulos irrelevantes como perderse títulos que te gustarían.  
c) Un modelo que filtra currículums para una búsqueda de empleo muy competitiva (pocos candidatos calificados entre muchos).

<details>
<summary>Resultado esperado</summary>

**a) Detección de incendios:** priorizarías **recall**. El error grave es no detectar un incendio real (FN). Un FP (falsa alarma) molesta, pero un FN puede costar vidas.

**b) Recomendaciones de contenido:** priorizarías **F1**. Querés balance: no molestar al usuario con títulos malos (precision alta) y no perderte títulos que le gustarían (recall alto).

**c) Filtrado de currículums con pocos calificados:** priorizarías **precision**. Si los candidatos calificados son escasos, no querés desperdiciar tiempo de entrevistas en candidatos que no cumplen el perfil (FP alto). El costo de un FN también existe, pero es menor que el costo operativo de entrevistar a muchos candidatos inadecuados.

</details>

## 4. Clímax: la trampa del 99% de accuracy en detección de fraude

Imaginemos que tenemos:

- **100 transacciones** en total.
- **99 son legítimas**, **1 es fraude** (caso extremadamente desbalanceado pero realista).
- Construimos un modelo "vago" que **siempre predice 'no es fraude'**, sin importar la transacción.

¿Cuánto da accuracy? ¿Cuánto da recall? Calculemos.

In [ ]:
# Caso fraude: el modelo dice "no fraude" para todas las transacciones.
# Convención: 1 = fraude (el positivo), 0 = legítima.

# Construimos los vectores manualmente para que se vea claro lo que pasa.
y_real_fraude = np.array([0]*99 + [1]*1)        # 99 legítimas + 1 fraude
y_pred_fraude = np.array([0]*100)                # el modelo dice "no fraude" a todas

cm_fraude = confusion_matrix(y_real_fraude, y_pred_fraude, labels=[0, 1])
tn_f, fp_f, fn_f, tp_f = cm_fraude.ravel()

print(f"True Negatives  (TN): {tn_f}")
print(f"False Positives (FP): {fp_f}")
print(f"False Negatives (FN): {fn_f} ← el único fraude real, no lo atrapamos")
print(f"True Positives  (TP): {tp_f}")
print()

# Calculamos accuracy.
acc_f = (tp_f + tn_f) / (tp_f + tn_f + fp_f + fn_f)
print(f"Accuracy = ({tp_f} + {tn_f}) / 100 = {acc_f:.2%}  ← parece BUENÍSIMA")

# Recall: TP / (TP + FN).
recall_f = tp_f / (tp_f + fn_f) if (tp_f + fn_f) > 0 else float("nan")
print(f"Recall   = {tp_f} / ({tp_f} + {fn_f}) = {recall_f:.2%}  ← el modelo no atrapa NINGÚN fraude")

# Precision: TP / (TP + FP). En este caso es 0/0 → indefinido.
if (tp_f + fp_f) == 0:
    print(f"Precision = {tp_f} / ({tp_f} + {fp_f}) = INDEFINIDO (no se puede dividir por 0)")
else:
    prec_f = tp_f / (tp_f + fp_f)
    print(f"Precision = {prec_f:.2%}")

**Conclusión clave:**

Un modelo que **NUNCA atrapa un fraude** tiene **99% de accuracy**. Es decir: la accuracy alta puede ser una mentira cuando los datos están desbalanceados. En este caso, **recall = 0** revela inmediatamente que el modelo es inútil para detectar fraude.

**Por eso accuracy ≠ precision, y por eso necesitamos las 4 métricas — no solo una.**

**Regla a recordar:** la métrica que uses depende del COSTO de cada tipo de error. No hay métrica universal.

## Cierre de la lección

**Lo que vimos:**

- **RMSE** mide qué tan lejos quedaron las predicciones de la realidad en regresión. Siempre se interpreta contra el rango del output.
- La **confusion matrix** muestra los 4 tipos de resultado en clasificación: TP, TN, FP, FN. Es lo CRUDO.
- **Accuracy / precision / recall / F1** son cuatro formas distintas de RESUMIR la confusion matrix en un número. Cada una responde una pregunta distinta.
- En datasets desbalanceados, **accuracy puede mentir**. Hay que mirar precision y recall para ver la verdad.
- La métrica a optimizar depende del **costo de cada tipo de error** en TU contexto.